# Propagate pseudotime to UTR datasets

Joins the per-lineage DPT pseudotimes computed in `05_260502-pseudotime_inhibitory.ipynb` and `04_260502-pseudotime_excitatory.ipynb` into:

1. The full UMI dataset (both lineage columns).
2. The three UTR datasets (raw counts, percentages, percentages+harmony), via the cell-key mapping `UMI: <seq>-<batch>` to `UTR: <seq>-1_<batch>`.

For the raw UTR file (`merged_bw12_above10cts_apa_named.h5ad`), this notebook also propagates all annotation columns from `merged_bw12_above10cts_apa_named_percentages_annotated_fix.h5ad` (these two files share `obs_names` exactly), turning the raw UTR file into the canonical input for the Dirichlet-multinomial regression in `260317-dirichlet_multinomial_analysis_optimized.R`.

Output column convention: two NaN-padded columns `dpt_pseudotime_inhibitory` and `dpt_pseudotime_excitatory`. The R script expects a single `dpt_pseudotime`, which is constructed by the per-lineage extraction helper at the end of this notebook.

Cells with no UMI counterpart (300 UTR cells that failed UMI QC) and cells excluded from the lineage pseudotime computation (e.g. Astrocytes, which are dropped from both lineages mirroring the published Knoblich CHOOSE pipeline at github.com/quadbio/ASD_CHOOSE/blob/main/scripts/fig2.R) keep NaN in both columns; the R DirMult script drops NaN-pseudotime cells automatically.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "environment" / "apa-python.yml").is_file()
    )
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/organoids_knoblich/UTR")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
sc.settings.verbosity = 1

BASEPATH = f"{DATA_ROOT}/3utr/organoids/knoblich"
COUNTS_PATH = f"{BASEPATH}/cellranger"
UTR_PATH = f"{BASEPATH}/utrcalling"

COUNTS_FILE = f"{COUNTS_PATH}/aggregated_annotated_fix_filtered_normalized.h5ad"
INHIB_DPT = f"{COUNTS_PATH}/aggregated_annotated_fix_filtered_normalized_inhibitory_dpt.h5ad"
EXCIT_DPT = f"{COUNTS_PATH}/aggregated_annotated_fix_filtered_normalized_excitatory_dpt.h5ad"
OUT_UMI_FILE = f"{COUNTS_PATH}/aggregated_annotated_fix_filtered_normalized_dpt.h5ad"

UTR_RAW = f"{UTR_PATH}/merged_bw12_above10cts_apa_named.h5ad"
UTR_PCT_FIX = f"{UTR_PATH}/merged_bw12_above10cts_apa_named_percentages_annotated_fix.h5ad"
UTR_PCT_HARM = f"{UTR_PATH}/merged_bw12_above10cts_apa_named_percentages_annotated_harmony.h5ad"

OUT_UTR_RAW = f"{UTR_PATH}/merged_bw12_above10cts_apa_named_annotated_fix_pt.h5ad"
OUT_UTR_PCT_FIX = f"{UTR_PATH}/merged_bw12_above10cts_apa_named_percentages_annotated_fix_pt.h5ad"
OUT_UTR_PCT_HARM = (
    f"{UTR_PATH}/merged_bw12_above10cts_apa_named_percentages_annotated_harmony_pt.h5ad"
)

print(f"Inputs:\n  {COUNTS_FILE}\n  {INHIB_DPT}\n  {EXCIT_DPT}")
print(f"  {UTR_RAW}\n  {UTR_PCT_FIX}\n  {UTR_PCT_HARM}\n")
print(f"Outputs:\n  {OUT_UMI_FILE}\n  {OUT_UTR_RAW}\n  {OUT_UTR_PCT_FIX}\n  {OUT_UTR_PCT_HARM}")

## Cell-key helpers

UMI obs_names follow `<barcode>-<batch_idx>` (cellranger aggr convention). UTR obs_names follow `<barcode>-1_<batch_number>` (the `-1` comes from the per-sample cellranger output preserved during merging, the `_<batch>` suffix was added by the merge step). Both formats encode the same cell when their `(barcode, batch_int)` keys match.


In [ ]:
def umi_to_key(name):
    seq, b = name.rsplit("-", 1)
    return (seq, int(b))


def utr_to_key(name):
    pre, b = name.rsplit("_", 1)
    seq = pre[:-2] if pre.endswith("-1") else pre
    return (seq, int(b))


# Quick sanity check on a few barcodes
print("UMI examples:")
for ex in ["AAACCCAGTCACTGAT-1", "TTTGGTTAGTAACGAT-22"]:
    print(f"  {ex} -> {umi_to_key(ex)}")
print("UTR examples:")
for ex in ["AAACCCAAGAATCGCG-1_6", "TTTGTTGTCTTGCGCT-1_9"]:
    print(f"  {ex} -> {utr_to_key(ex)}")

## Load data


In [ ]:
adata_full = sc.read_h5ad(COUNTS_FILE)
adata_inh = sc.read_h5ad(INHIB_DPT)
adata_exc = sc.read_h5ad(EXCIT_DPT)
print(f"Full UMI: {adata_full.shape}")
print(
    f"Inhibitory checkpoint: {adata_inh.shape}, dpt range "
    f"[{adata_inh.obs['dpt_pseudotime'].min():.3f}, "
    f"{adata_inh.obs['dpt_pseudotime'].max():.3f}]"
)
print(
    f"Excitatory checkpoint: {adata_exc.shape}, dpt range "
    f"[{adata_exc.obs['dpt_pseudotime'].min():.3f}, "
    f"{adata_exc.obs['dpt_pseudotime'].max():.3f}]"
)

In [ ]:
adata_utr_raw = sc.read_h5ad(UTR_RAW)
adata_utr_pct_fix = sc.read_h5ad(UTR_PCT_FIX)
adata_utr_pct_harm = sc.read_h5ad(UTR_PCT_HARM)
print(f"UTR raw: {adata_utr_raw.shape}, obs cols: {list(adata_utr_raw.obs.columns)}")
print(
    f"UTR percentages_fix: {adata_utr_pct_fix.shape}, obs cols: {list(adata_utr_pct_fix.obs.columns)}"
)
print(
    f"UTR percentages_harmony: {adata_utr_pct_harm.shape}, obs cols: {list(adata_utr_pct_harm.obs.columns)}"
)

In [ ]:
adata_inh

In [ ]:
sc.pl.umap(
    adata_inh,
    color=["dpt_pseudotime", "celltype", "induced"],
    title=["Inhibitory DPT pseudotime", "Celltype", "Induced"],
)

In [ ]:
sc.pl.umap(
    adata_exc,
    color=["dpt_pseudotime", "celltype", "induced"],
    title=["Excitatory DPT pseudotime", "Celltype", "Induced"],
)

## Build pseudotime DataFrame keyed by `(barcode, batch)`


In [ ]:
umi_keys = [umi_to_key(n) for n in adata_full.obs_names]
pt_df = pd.DataFrame(
    index=pd.MultiIndex.from_tuples(umi_keys, names=["seq", "batch"]),
    columns=["dpt_pseudotime_inhibitory", "dpt_pseudotime_excitatory"],
    dtype=float,
)

inh_keys = [umi_to_key(n) for n in adata_inh.obs_names]
exc_keys = [umi_to_key(n) for n in adata_exc.obs_names]
pt_df.loc[inh_keys, "dpt_pseudotime_inhibitory"] = adata_inh.obs["dpt_pseudotime"].values
pt_df.loc[exc_keys, "dpt_pseudotime_excitatory"] = adata_exc.obs["dpt_pseudotime"].values

n_inh = pt_df["dpt_pseudotime_inhibitory"].notna().sum()
n_exc = pt_df["dpt_pseudotime_excitatory"].notna().sum()
print(f"pt_df shape: {pt_df.shape}")
print(f"Non-NaN inhibitory: {n_inh:,} (expected {adata_inh.n_obs:,})")
print(f"Non-NaN excitatory: {n_exc:,} (expected {adata_exc.n_obs:,})")
assert n_inh == adata_inh.n_obs, "Inhibitory key mismatch."
assert n_exc == adata_exc.n_obs, "Excitatory key mismatch."
assert pt_df.index.is_unique, "UMI keys are not unique; check obs_names."
print("Pseudotime DataFrame built correctly.")

## Update full UMI and save


In [ ]:
adata_full.obs["dpt_pseudotime_inhibitory"] = pt_df.loc[
    umi_keys, "dpt_pseudotime_inhibitory"
].values
adata_full.obs["dpt_pseudotime_excitatory"] = pt_df.loc[
    umi_keys, "dpt_pseudotime_excitatory"
].values
adata_full.write_h5ad(OUT_UMI_FILE)
print(f"Saved: {OUT_UMI_FILE}")
print(f"  inhibitory non-NaN: {adata_full.obs['dpt_pseudotime_inhibitory'].notna().sum():,}")
print(f"  excitatory non-NaN: {adata_full.obs['dpt_pseudotime_excitatory'].notna().sum():,}")

## Propagate to UTR files


In [ ]:
def add_pseudotime(adata_utr, pt_df):
    keys = [utr_to_key(n) for n in adata_utr.obs_names]
    keys_idx = pd.MultiIndex.from_tuples(keys, names=["seq", "batch"])
    reidx = pt_df.reindex(keys_idx)
    adata_utr.obs["dpt_pseudotime_inhibitory"] = reidx["dpt_pseudotime_inhibitory"].values
    adata_utr.obs["dpt_pseudotime_excitatory"] = reidx["dpt_pseudotime_excitatory"].values
    return adata_utr


for label, ad in [
    ("UTR raw", adata_utr_raw),
    ("UTR percentages_fix", adata_utr_pct_fix),
    ("UTR percentages_harmony", adata_utr_pct_harm),
]:
    add_pseudotime(ad, pt_df)
    n_inh_u = ad.obs["dpt_pseudotime_inhibitory"].notna().sum()
    n_exc_u = ad.obs["dpt_pseudotime_excitatory"].notna().sum()
    n_both_nan = (
        ad.obs[["dpt_pseudotime_inhibitory", "dpt_pseudotime_excitatory"]].isna().all(axis=1).sum()
    )
    print(f"{label}: n_inh={n_inh_u:,}  n_exc={n_exc_u:,}  both_NaN={n_both_nan:,}")

## Propagate full annotations to raw UTR

The raw UTR file currently has only `batch_number` and `sample_id`. The percentages_fix file has the same `obs_names` in the same order, so we can copy annotation columns by direct value-copy.


In [ ]:
assert adata_utr_raw.obs_names.equals(adata_utr_pct_fix.obs_names), (
    "UTR raw and UTR percentages_fix have different obs_names; cannot direct-copy."
)

annotation_cols = [
    "nCount_RNA",
    "nFeature_RNA",
    "percent.mito",
    "percent.ribo",
    "gRNA",
    "induced",
    "celltype",
    "lineage",
]

for col in annotation_cols:
    if col not in adata_utr_pct_fix.obs.columns:
        print(f"WARNING: '{col}' missing from percentages_fix obs; skipping.")
        continue
    adata_utr_raw.obs[col] = adata_utr_pct_fix.obs[col].values

print(f"Raw UTR obs columns now: {list(adata_utr_raw.obs.columns)}")

# Convert raw UTR X from int sparse to float64 sparse so the R anndata package
# can construct dgRMatrix (dgRMatrix requires double-typed x slot). The
# percentages files are already float by construction.
import numpy as np

if adata_utr_raw.X.dtype != np.float64:
    print(f"Converting raw UTR X: {adata_utr_raw.X.dtype} -> float64")
    adata_utr_raw.X = adata_utr_raw.X.astype(np.float64)

## Sanity checks


In [ ]:
# Cell-count assertions per file
for label, ad in [
    ("UTR raw", adata_utr_raw),
    ("UTR percentages_fix", adata_utr_pct_fix),
    ("UTR percentages_harmony", adata_utr_pct_harm),
]:
    n_inh_u = ad.obs["dpt_pseudotime_inhibitory"].notna().sum()
    n_exc_u = ad.obs["dpt_pseudotime_excitatory"].notna().sum()
    n_both_nan = (
        ad.obs[["dpt_pseudotime_inhibitory", "dpt_pseudotime_excitatory"]].isna().all(axis=1).sum()
    )
    assert n_inh_u == adata_inh.n_obs, (
        f"{label}: inhibitory count mismatch {n_inh_u} vs {adata_inh.n_obs}"
    )
    assert n_exc_u == adata_exc.n_obs, (
        f"{label}: excitatory count mismatch {n_exc_u} vs {adata_exc.n_obs}"
    )
    # Both-NaN cells = (UTR cells absent from UMI) + (UMI cells excluded from both lineage
    # checkpoints, e.g. Astrocytes, dropped as in Li et al. 2023). Total = ad.n_obs - n_inh_u - n_exc_u
    # because every UMI cell is in at most one lineage checkpoint.
    expected_both_nan = ad.n_obs - n_inh_u - n_exc_u
    assert n_both_nan == expected_both_nan, (
        f"{label}: both-NaN count {n_both_nan} != expected {expected_both_nan}"
    )
    n_utr_only = ad.n_obs - adata_full.n_obs  # 300 by construction
    n_lineage_excluded = n_both_nan - n_utr_only
    print(
        f"{label}: OK  (UTR-only cells: {n_utr_only}, "
        f"UMI cells excluded from both lineages [e.g. Astrocytes]: {n_lineage_excluded})"
    )

# Raw UTR celltype must match percentages_fix celltype after propagation
ct_raw = adata_utr_raw.obs["celltype"].astype(str)
ct_pct = adata_utr_pct_fix.obs["celltype"].astype(str)
assert (ct_raw == ct_pct).all(), "Raw UTR celltype does not match percentages_fix."
print("Raw UTR celltype matches percentages_fix celltype.")

## Add `super_batch` (3-level capture batch)

Derived from the first 3 characters of `sample` / `sample_id`, matching the GEX capture batch in `samples.tsv` of the Snakemake workflow (`01_snakemake_workflow`):

- `137` = batch1 (8 libraries, all 4OHT_induced)
- `206` = batch3 (10 libraries, mixed induction)
- `213` = batch2 (4 libraries, mixed induction)

This 3-level grouping captures the dominant technical batch effect while preserving within-super-batch induction variation (mixed in 206 and 213). It is used as a sensitivity covariate in the Dirichlet-multinomial regression.


In [ ]:
def add_super_batch(adata, sample_col):
    s = adata.obs[sample_col].astype(str).str[:3]
    adata.obs["super_batch"] = pd.Categorical(s, categories=sorted(s.unique()))
    return adata


add_super_batch(adata_full, "sample")
add_super_batch(adata_utr_raw, "sample_id")
add_super_batch(adata_utr_pct_fix, "sample_id")
add_super_batch(adata_utr_pct_harm, "sample_id")

print("super_batch counts:")
for label, ad in [
    ("full UMI", adata_full),
    ("UTR raw", adata_utr_raw),
    ("UTR percentages_fix", adata_utr_pct_fix),
    ("UTR percentages_harmony", adata_utr_pct_harm),
]:
    counts = ad.obs["super_batch"].value_counts().sort_index().to_dict()
    print(f"  {label}: {counts}")

## Save UTR files


In [ ]:
adata_utr_raw.write_h5ad(OUT_UTR_RAW)
print(f"Saved: {OUT_UTR_RAW}  ({adata_utr_raw.shape})")

adata_utr_pct_fix.write_h5ad(OUT_UTR_PCT_FIX)
print(f"Saved: {OUT_UTR_PCT_FIX}  ({adata_utr_pct_fix.shape})")

adata_utr_pct_harm.write_h5ad(OUT_UTR_PCT_HARM)
print(f"Saved: {OUT_UTR_PCT_HARM}  ({adata_utr_pct_harm.shape})")

In [ ]:
sc.pl.umap(
    adata_utr_pct_harm,
    color=["celltype", "dpt_pseudotime_inhibitory", "dpt_pseudotime_excitatory"],
    ncols=1,
    size=20,
)

In [ ]:
sc.pl.pca(
    adata_utr_pct_harm,
    color=["celltype", "dpt_pseudotime_inhibitory", "dpt_pseudotime_excitatory"],
    ncols=1,
    size=20,
)

## Per-lineage extraction for the R DirMult script

The R script `260317-dirichlet_multinomial_analysis_optimized.R` reads `obs.dpt_pseudotime` (single column) and `obs.batch_number` and `var.utr_name`, and drops cells with NaN pseudotime. Run the helper below to write per-lineage subsets with the lineage-specific pseudotime renamed to `dpt_pseudotime`.


In [ ]:
def extract_for_dirmult(adata_utr_pt_path, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    a = sc.read_h5ad(adata_utr_pt_path)
    written = []
    for lineage_name, tag in [
        ("Inhibitory neurons", "inhibitory"),
        ("Excitatory neurons", "excitatory"),
    ]:
        ad = a[a.obs["lineage"] == lineage_name].copy()
        col = f"dpt_pseudotime_{tag}"
        ad.obs["dpt_pseudotime"] = ad.obs[col]
        ad = ad[ad.obs["dpt_pseudotime"].notna()].copy()
        out_path = out_dir / f"{Path(adata_utr_pt_path).stem}_{tag}_for_dirmult.h5ad"
        ad.write_h5ad(out_path)
        written.append(out_path)
        print(f"  {tag}: {ad.shape} -> {out_path}")
    return written


# Optional: uncomment to run now
# extract_for_dirmult(OUT_UTR_RAW, f"{UTR_PATH}/dirmult_inputs")

### R script invocation

```
Rscript $M26_DATA_ROOT/notebooks/3UTR/Wildtype_mice/2026/02-optimized_method_1/260317-dirichlet_multinomial_analysis_optimized.R \
    <path>/merged_bw12_above10cts_apa_named_annotated_fix_pt_inhibitory_for_dirmult.h5ad
```

(plus any flags the script accepts; check its argparser before running)
